# R1

In [34]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-02.parquet"

df = pd.read_parquet(url)

print(df.shape)
print(df.head())

(3399866, 20)
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         7  2026-02-01 00:05:57   2026-02-01 00:05:57              1.0   
1         7  2026-02-01 00:35:58   2026-02-01 00:35:58              1.0   
2         2  2026-02-01 00:08:41   2026-02-01 00:39:32              1.0   
3         1  2026-02-01 00:29:06   2026-02-01 00:41:04              0.0   
4         1  2026-02-01 00:53:52   2026-02-01 01:11:21              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.94         1.0                  N           107           170   
1           1.93         1.0                  N           234           141   
2           9.99         1.0                  N           138            68   
3           1.70         1.0                  N           209            13   
4           3.70         1.0                  N           249           229   

   payment_type  fare_amount  extra  mta_tax  tip_amount  to

In [35]:
print("\nMissing Percentage:")
print((df.isnull().mean()*100).sort_values(ascending=False))

print("\nUnique Values:")
print(df.nunique().sort_values(ascending=False))


Missing Percentage:
passenger_count          30.098745
congestion_surcharge     30.098745
store_and_fwd_flag       30.098745
RatecodeID               30.098745
Airport_fee              30.098745
tpep_dropoff_datetime     0.000000
VendorID                  0.000000
tpep_pickup_datetime      0.000000
DOLocationID              0.000000
payment_type              0.000000
trip_distance             0.000000
PULocationID              0.000000
extra                     0.000000
fare_amount               0.000000
mta_tax                   0.000000
tip_amount                0.000000
improvement_surcharge     0.000000
tolls_amount              0.000000
total_amount              0.000000
cbd_congestion_fee        0.000000
dtype: float64

Unique Values:
tpep_pickup_datetime     1587963
tpep_dropoff_datetime    1587241
total_amount               19930
fare_amount                12616
trip_distance               4719
tip_amount                  4210
tolls_amount                1169
DOLocationID     

# R2

In [36]:
import pandas as pd
import numpy as np

audit = []

for col in df.columns:

    row = {}

    row["column"] = col
    row["dtype"] = str(df[col].dtype)
    row["% missing"] = round(df[col].isnull().mean()*100, 2)
    row["distinct"] = df[col].nunique()

    # Min / Max
    if pd.api.types.is_numeric_dtype(df[col]):
        row["min/max"] = f"{df[col].min()} / {df[col].max()}"

        # Outlier Evidence using IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = ((df[col] < lower) | (df[col] > upper)).sum()

        row["outliers"] = outliers

        if outliers > 0:
            row["proposed treatment"] = "Keep + RobustScaler / Log Transform"
        else:
            row["proposed treatment"] = "No action required"

    else:
        row["min/max"] = str(df[col].value_counts().head(3).to_dict())
        row["outliers"] = "N/A"

        if row["distinct"] <= 10:
            row["proposed treatment"] = "One-Hot Encoding"
        else:
            row["proposed treatment"] = "Group Rare + One-Hot Encoding"

    audit.append(row)

audit_df = pd.DataFrame(audit)

audit_df

,column,dtype,% missing,distinct,min/max,outliers,proposed treatment
0,VendorID,int32,0.0,4,1 / 7,683449,Keep + RobustScaler / Log Transform
1,tpep_pickup_datetime,datetime64[us],0.0,1587963,"{Timestamp('2026-02-07 22:48:00'): 14, Timesta...",N/A,Group Rare + One-Hot Encoding
2,tpep_dropoff_datetime,datetime64[us],0.0,1587241,"{Timestamp('2026-02-26 19:00:08'): 14, Timesta...",N/A,Group Rare + One-Hot Encoding
3,passenger_count,float64,30.1,9,0.0 / 9.0,401854,Keep + RobustScaler / Log Transform
4,trip_distance,float64,0.0,4719,0.0 / 328522.2,380461,Keep + RobustScaler / Log Transform
5,RatecodeID,float64,30.1,7,1.0 / 99.0,213909,Keep + RobustScaler / Log Transform
6,store_and_fwd_flag,object,30.1,2,"{'N': 2374445, 'Y': 2104}",N/A,One-Hot Encoding
7,PULocationID,int32,0.0,259,1 / 265,0,No action required
8,DOLocationID,int32,0.0,261,1 / 265,0,No action required
9,payment_type,int64,0.0,5,0 / 4,51272,Keep + RobustScaler / Log Transform


# R3

In [37]:
missing_cols = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

In [38]:
for col in missing_cols:
    print("\n" + "="*60)
    print("COLUMN:", col)

    temp = df.copy()
    temp[col + "_missing"] = temp[col].isna()

    print(
        temp.groupby(col + "_missing")["trip_distance"]
        .mean()
    )

    print(
        temp.groupby(col + "_missing")["fare_amount"]
        .mean()
    )

    print(
        temp.groupby(col + "_missing")["total_amount"]
        .mean()
    )


COLUMN: passenger_count
passenger_count_missing
False     3.346450
True     12.966495
Name: trip_distance, dtype: float64
passenger_count_missing
False    19.243154
True     27.228753
Name: fare_amount, dtype: float64
passenger_count_missing
False    28.576176
True     33.682954
Name: total_amount, dtype: float64

COLUMN: RatecodeID
RatecodeID_missing
False     3.346450
True     12.966495
Name: trip_distance, dtype: float64
RatecodeID_missing
False    19.243154
True     27.228753
Name: fare_amount, dtype: float64
RatecodeID_missing
False    28.576176
True     33.682954
Name: total_amount, dtype: float64

COLUMN: store_and_fwd_flag
store_and_fwd_flag_missing
False     3.346450
True     12.966495
Name: trip_distance, dtype: float64
store_and_fwd_flag_missing
False    19.243154
True     27.228753
Name: fare_amount, dtype: float64
store_and_fwd_flag_missing
False    28.576176
True     33.682954
Name: total_amount, dtype: float64

COLUMN: congestion_surcharge
congestion_surcharge_missing
F

In [39]:

missing_cols = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

# Create missing indicators
for col in missing_cols:
    df[col + "_missing"] = df[col].isna().astype(int)


temp_df = df.copy()


num_cols = [
    "passenger_count",
    "congestion_surcharge",
    "Airport_fee"
]


cat_cols = [
    "RatecodeID",
    "store_and_fwd_flag"
]

print("NUMERIC COLUMNS")

for col in num_cols:

    before_mean = df[col].mean()
    before_median = df[col].median()
    before_std = df[col].std()

    # Median Imputation tempariraly
    temp_df[col] = temp_df[col].fillna(temp_df[col].median())

    after_mean = temp_df[col].mean()
    after_median = temp_df[col].median()
    after_std = temp_df[col].std()

    print(f"\nColumn: {col}")

    print("Before Imputation")
    print("Mean   :", round(before_mean, 4))
    print("Median :", round(before_median, 4))
    print("Std    :", round(before_std, 4))

    print("\nAfter Imputation")
    print("Mean   :", round(after_mean, 4))
    print("Median :", round(after_median, 4))
    print("Std    :", round(after_std, 4))

print("CATEGORICAL COLUMNS")

for col in cat_cols:
    before_missing = df[col].isna().sum()
    temp_df[col] = temp_df[col].fillna(
        temp_df[col].mode()[0]
    )

    after_missing = temp_df[col].isna().sum()

    print(f"\nColumn: {col}")
    print("Missing Before:", before_missing)
    print("Missing After :", after_missing)
    print("Imputation Used: Mode")

NUMERIC COLUMNS

Column: passenger_count
Before Imputation
Mean   : 1.2299
Median : 1.0
Std    : 0.6347

After Imputation
Mean   : 1.1607
Median : 1.0
Std    : 0.5411

Column: congestion_surcharge
Before Imputation
Mean   : 2.1726
Median : 2.5
Std    : 0.9073

After Imputation
Mean   : 2.2711
Median : 2.5
Std    : 0.7733

Column: Airport_fee
Before Imputation
Mean   : 0.143
Median : 0.0
Std    : 0.5204

After Imputation
Mean   : 0.1
Median : 0.0
Std    : 0.44
CATEGORICAL COLUMNS

Column: RatecodeID
Missing Before: 1023317
Missing After : 0
Imputation Used: Mode

Column: store_and_fwd_flag
Missing Before: 1023317
Missing After : 0
Imputation Used: Mode


# R4

In [40]:
numeric_cols = df.select_dtypes(include=['int64','int32','float64']).columns

skew_table = pd.DataFrame({
    "column": numeric_cols,
    "skewness": [df[col].skew() for col in numeric_cols]
})

skew_table.sort_values("skewness", ascending=False)

,column,skewness
2,trip_distance,267.401897
10,tip_amount,5.805544
11,tolls_amount,5.353912
15,Airport_fee,5.053492
0,VendorID,4.537289
3,RatecodeID,4.534159
1,passenger_count,3.372040
7,fare_amount,3.204593
13,total_amount,2.916229
8,extra,1.908797


In [41]:
for col in numeric_cols:
    print("\n", col)
    print(df[col].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))


 VendorID
count    3.399866e+06
mean     1.878228e+00
std      7.035353e-01
min      1.000000e+00
1%       1.000000e+00
5%       1.000000e+00
25%      2.000000e+00
50%      2.000000e+00
75%      2.000000e+00
95%      2.000000e+00
99%      7.000000e+00
max      7.000000e+00
Name: VendorID, dtype: float64

 passenger_count
count    2.376549e+06
mean     1.229941e+00
std      6.347402e-01
min      0.000000e+00
1%       1.000000e+00
5%       1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
95%      2.000000e+00
99%      4.000000e+00
max      9.000000e+00
Name: passenger_count, dtype: float64

 trip_distance
count    3.399866e+06
mean     6.241963e+00
std      6.336499e+02
min      0.000000e+00
1%       0.000000e+00
5%       2.200000e-01
25%      1.000000e+00
50%      1.800000e+00
75%      3.680000e+00
95%      1.205000e+01
99%      1.941000e+01
max      3.285222e+05
Name: trip_distance, dtype: float64

 RatecodeID
count    2.376549e+06
mean     5.250941e+00
s

# R5

In [42]:
print("R5 - ENCODING AUDIT")

categorical_cols = [
    "VendorID",
    "payment_type",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID"
]

for col in categorical_cols:
    unique_count = df[col].nunique(dropna=True)

    print(f"\nColumn: {col}")
    print(f"Unique Categories: {unique_count}")

    if unique_count <= 10:
        print("Encoding Choice: One-Hot Encoding")
        print("Reason: Low-cardinality categorical feature.")
    else:
        print("Encoding Choice: Group Rare Categories + One-Hot Encoding")
        print("Reason: High-cardinality categorical feature.")

R5 - ENCODING AUDIT

Column: VendorID
Unique Categories: 4
Encoding Choice: One-Hot Encoding
Reason: Low-cardinality categorical feature.

Column: payment_type
Unique Categories: 5
Encoding Choice: One-Hot Encoding
Reason: Low-cardinality categorical feature.

Column: RatecodeID
Unique Categories: 7
Encoding Choice: One-Hot Encoding
Reason: Low-cardinality categorical feature.

Column: store_and_fwd_flag
Unique Categories: 2
Encoding Choice: One-Hot Encoding
Reason: Low-cardinality categorical feature.

Column: PULocationID
Unique Categories: 259
Encoding Choice: Group Rare Categories + One-Hot Encoding
Reason: High-cardinality categorical feature.

Column: DOLocationID
Unique Categories: 261
Encoding Choice: Group Rare Categories + One-Hot Encoding
Reason: High-cardinality categorical feature.


In [43]:
! touch pipeline.py



In [44]:
%%writefile pipeline.py

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LinearRegression


def build_pipeline():

    numeric_cols = [
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "extra",
        "Airport_fee",
        "congestion_surcharge",
        "cbd_congestion_fee"
    ]

    categorical_cols = [
        "VendorID",
        "payment_type",
        "RatecodeID",
        "store_and_fwd_flag",
        "PULocationID",
        "DOLocationID"
    ]

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, numeric_cols),
        ("cat", cat_pipeline, categorical_cols)
    ])

    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    return model_pipeline

Overwriting pipeline.py


# R6

In [45]:
from pipeline import build_pipeline

model_pipeline = build_pipeline()

print(model_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['trip_distance',
                                                   'fare_amount', 'tip_amount',
                                                   'tolls_amount', 'extra',
                                                   'Airport_fee',
                                                   'congestion_surcharge',
                                                   'cbd_congestion_fee']),
                                                 ('cat',
                                                  Pipeline(steps=[('im